# My Personal Technical Tutor
Ask any technical question and choose whether to answer using **GPT** (via OpenRouter) or **Llama** (via Ollama).

Includes an **AI Guardrail Validation Step** powered by OpenRouter's free model router (`openrouter/free`).

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display, update_display

In [ ]:
MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

# NEW: Model for AI Guardrail input validation
MODEL_GUARDRAIL = 'openrouter/free'

load_dotenv(override=True)
api_key = os.getenv('OPENROUTER_API_KEY')

# Client for OpenRouter (GPT & AI Guardrail)
openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1", 
    api_key=api_key
)

# Client for local Ollama (Llama)
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

## Section 1: AI Guardrail (Input Validation)
This function acts as a gatekeeper using `openrouter/free` to check if the user's input is a valid, complete technical prompt or random nonsense.

In [ ]:
# CHANGED: Updated prompt to accept valid queries without '?' (e.g. 'what is jupyter') while rejecting gibberish
def validate_question_with_ai(client, question_text: str) -> bool:
    """Uses OpenRouter free model to validate if the input is a meaningful technical question/query."""
    words = question_text.strip().split()
    
    # Pre-check requiring at least 3 words
    if len(words) < 3:
        print("Input too short! Please enter at least 3 words.")
        return False
        
    try:
        response = client.chat.completions.create(
            model=MODEL_GUARDRAIL,
            messages=[
                {
                    "role": "system", 
                    "content": (
                        "You are an input validator. Your job is to check if the user's input is a technical question or coding prompt.\n"
                        "Respond ONLY with 'YES' if the text is asking a question or requesting information about a technology/coding topic (e.g., 'what is jupyter', 'explain recursion', 'how does python work'), EVEN IF it lacks punctuation or a question mark.\n"
                        "Respond ONLY with 'NO' if the text is random gibberish (e.g., 'abcsd', 'asdfghjkl'), keyboard mashing, or complete nonsense."
                    )
                },
                {"role": "user", "content": question_text}
            ],
            max_tokens=5
        )
        result = response.choices[0].message.content.strip().upper()
        return "YES" in result
    except Exception as e:
        print(f"(Guardrail warning: {e}. Falling back to word count check)")
        return len(words) >= 3


## Section 2: User Input & Model Selection
Prompts the user for a question (type 'exit' to cancel anytime), runs it through the AI Guardrail until a valid question is entered, then asks the user to select the model provider.

In [ ]:
def get_valid_question(client):
    """Returns the validated question string, or None if user cancels."""
    while True:
        user_input = input("Enter your question (or 'exit' to cancel): ").strip()
        if user_input.lower() in ['exit', 'quit', 'q', 'cancel']:
            print("Operation cancelled by user.")
            return None
        print(f"Validating question with AI Guardrail...")
        if validate_question_with_ai(client, user_input):
            print("Input validated successfully!\n")
            return user_input
        else:
            print("Invalid input! Please enter a meaningful technical question.\n")


def select_provider():
    """Returns (model, client, provider_name), or None if user cancels."""
    options = {
        '1': (MODEL_GPT, openai_client, "OpenRouter (GPT-4o-mini)"),
        'gpt': (MODEL_GPT, openai_client, "OpenRouter (GPT-4o-mini)"),
        '2': (MODEL_LLAMA, ollama_client, "Ollama (Llama 3.2)"),
        'llama': (MODEL_LLAMA, ollama_client, "Ollama (Llama 3.2)"),
    }
    while True:
        print("Select model provider to answer your question:")
        print("1. GPT (gpt-4o-mini via OpenRouter)")
        print("2. Llama (llama3.2 via Ollama)")
        choice = input("Enter 1 for GPT or 2 for Llama: ").strip().lower()
        if choice in ['exit', 'quit', 'q', 'cancel']:
            print("Model selection cancelled.")
            return None
        if choice in options:
            return options[choice]
        print("Invalid selection! Please enter 1 for GPT or 2 for Llama.\n")


## Section 3: Prompts & Response Streaming
Prepares system & user prompts and streams the response dynamically.

In [16]:
question = get_valid_question(openai_client)

if question:
    result = select_provider()
    if result:
        selected_model, selected_client, provider_name = result
        print(f"\nSelected Provider: {provider_name}")

        system_prompt = """You are a helpful technical tutor who provides beginner-friendly answers to questions about python code, 
software engineering, data science, and LLMs."""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Answer the following question with detailed explanations to a total beginner: {question}"}
        ]

        stream = selected_client.chat.completions.create(
            model=selected_model,
            messages=messages,
            stream=True
        )
        response = ""
        display_handle = display(Markdown(""), display_id=True)
        for chunk in stream:
            response += chunk.choices[0].delta.content or ''
            update_display(Markdown(response), display_id=display_handle.display_id)
    else:
        print("Execution skipped (model not selected).")
else:
    print("Execution skipped (no question set).")